In [2]:
# Run this cell only if you haven't installed the required packages
# NOTE: run in a notebook cell (prefix ! runs shell commands)
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm imbalanced-learn joblib openpyxl --quiet



print("Packages installed / already present.")


Packages installed / already present.


In [4]:
# 📌 Step 1: Import Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import matplotlib.pyplot as plt

# 📌 Step 2: Load Dataset
file_path = "data.xlsx"
data = pd.read_excel(file_path, sheet_name="Sheet1", engine="openpyxl")
#data = pd.read_excel(file_path, sheet_name="Sheet1")


# 📌 Step 3: Define Target (Threshold Rule: ≥4 symptoms = Cancer)
symptom_cols = [
    'Non_Healing_Ulcer', 'White_Red_Patches', 'Oral_Bleeding', 'Burning_Pain',
    'Hypertension', 'Diabetes', 'Anemia', 'Frequent_Infections',
    'Industrial_Exposure', 'Direct_Sunlight', 'Family_Cancer_History'
]

# Cancer = 1 if 3 or more symptoms/risk factors are present
data["Cancer"] = (data[symptom_cols].sum(axis=1) >= 4).astype(int)
                                                             

# 📌 Step 4: Split Features & Target
X = data.drop(columns=["Cancer"])
y = data["Cancer"]

# 📌 Step 5: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 📌 Step 6: Scale Features (for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 📌 Step 7: Initialize Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
}

# 📌 Step 8: Train & Evaluate Models
results = {}

for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    }
    print(f"\n{name} Classification Report:\n", classification_report(y_test, y_pred))

# 📌 Step 9: Compare Results
results_df = pd.DataFrame(results).T
print("\nModel Comparison:\n", results_df)

# 📌 Step 10: Plot Comparison
results_df.plot(kind="bar", figsize=(10,6))
plt.title("Model Performance Comparison (Rule 1)")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()

BadZipFile: File is not a zip file

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
import seaborn as sns

# 📌 Cross-Validation with Logistic Regression
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=cv, scoring="accuracy")

print("Cross-validation Accuracy Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

# 📌 Confusion Matrix (Random Forest as example)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No Cancer","Cancer"], yticklabels=["No Cancer","Cancer"])
plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


# 📌 ROC Curve (XGBoost as example)
xgb = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
xgb.fit(X_train, y_train)
y_pred_prob = xgb.predict_proba(X_test)[:,1]

fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
plt.plot(fpr, tpr, label=f"XGBoost (AUC = {roc_auc_score(y_test, y_pred_prob):.2f})")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


In [ ]:
# 📌 Step 11: ROC-AUC Curves
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import cross_val_score
import numpy as np

plt.figure(figsize=(8,6))

for name, model in models.items():
    if name == "Logistic Regression":
        y_proba = model.predict_proba(X_test_scaled)[:,1]
    else:
        y_proba = model.predict_proba(X_test)[:,1]
    
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.2f})")

plt.plot([0,1], [0,1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-AUC Curves")
plt.legend()
plt.show()



In [ ]:
# 📌 Step 12: Cross-Validation (5-fold Accuracy)
print("\nCross-Validation Results (5-fold Accuracy):")
for name, model in models.items():
    if name == "Logistic Regression":
        scores = cross_val_score(model, scaler.fit_transform(X), y, cv=5, scoring="accuracy")
    else:
        scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    print(f"{name}: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# 📌 Step 13: Feature Importance (Random Forest & XGBoost)
def plot_feature_importance(model, feature_names, title):
    importance = model.feature_importances_
    sorted_idx = np.argsort(importance)[::-1]
    plt.figure(figsize=(8,6))
    plt.bar(range(len(importance)), importance[sorted_idx], align="center")
    plt.xticks(range(len(importance)), np.array(feature_names)[sorted_idx], rotation=90)
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Random Forest Feature Importance
plot_feature_importance(models["Random Forest"], X.columns, "Random Forest Feature Importance")

# XGBoost Feature Importance
plot_feature_importance(models["XGBoost"], X.columns, "XGBoost Feature Importance")

In [ ]:
import joblib

# Save Logistic Regression model & scaler
joblib.dump(models["Logistic Regression"], "oral_cancer_model.pkl")
joblib.dump(scaler, "scaler.pkl")

In [ ]:
joblib.dump(list(X.columns), "feature_columns.pkl")



In [ ]:

!pip install streamlit
